In [1]:
import os
import json
import shutil
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("..").resolve()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"

COCO_DIR = DATA_RAW / "coco2017"
COCOBLUR_DIR = DATA_RAW / "cocoblur"

YOLO_DATASET_DIR = DATA_PROCESSED / "yolo_dataset"

BLUR_IMAGES_DIR = YOLO_DATASET_DIR / "blurred" / "images"
BLUR_LABELS_DIR = YOLO_DATASET_DIR / "blurred" / "labels"

SHARP_IMAGES_DIR = YOLO_DATASET_DIR / "sharp" / "images"
SHARP_LABELS_DIR = YOLO_DATASET_DIR / "sharp" / "labels"

RL_IMAGES_DIR = YOLO_DATASET_DIR / "richardson_lucy" / "images"
RL_LABELS_DIR = YOLO_DATASET_DIR / "richardson_lucy" / "labels"

for p in [
    BLUR_IMAGES_DIR,
    BLUR_LABELS_DIR,
    SHARP_IMAGES_DIR,
    SHARP_LABELS_DIR,
    RL_IMAGES_DIR,
    RL_LABELS_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

In [3]:
annotations_path = COCO_DIR / "annotations" / "instances_val2017.json"

with open(annotations_path, "r") as f:
    coco = json.load(f)

print(coco.keys())

dict_keys(['info', 'licenses', 'images', 'annotations', 'categories'])


In [4]:
images_df = pd.DataFrame(coco["images"])
annotations_df = pd.DataFrame(coco["annotations"])
categories_df = pd.DataFrame(coco["categories"])

print(images_df.shape)
print(annotations_df.shape)
print(categories_df.shape)

(5000, 8)
(36781, 7)
(80, 3)


In [5]:
categories_df[["id", "name"]].head(20)

,id,name
0,1,person
1,2,bicycle
2,3,car
3,4,motorcycle
4,5,airplane
5,6,bus
6,7,train
7,8,truck
8,9,boat
9,10,traffic light


In [6]:
category_id_to_index = {}
category_id_to_name = {}

for idx, row in categories_df.sort_values("id").reset_index(drop=True).iterrows():
    category_id_to_index[row["id"]] = idx
    category_id_to_name[row["id"]] = row["name"]

category_id_to_index

{1: 0,
 2: 1,
 3: 2,
 4: 3,
 5: 4,
 6: 5,
 7: 6,
 8: 7,
 9: 8,
 10: 9,
 11: 10,
 13: 11,
 14: 12,
 15: 13,
 16: 14,
 17: 15,
 18: 16,
 19: 17,
 20: 18,
 21: 19,
 22: 20,
 23: 21,
 24: 22,
 25: 23,
 27: 24,
 28: 25,
 31: 26,
 32: 27,
 33: 28,
 34: 29,
 35: 30,
 36: 31,
 37: 32,
 38: 33,
 39: 34,
 40: 35,
 41: 36,
 42: 37,
 43: 38,
 44: 39,
 46: 40,
 47: 41,
 48: 42,
 49: 43,
 50: 44,
 51: 45,
 52: 46,
 53: 47,
 54: 48,
 55: 49,
 56: 50,
 57: 51,
 58: 52,
 59: 53,
 60: 54,
 61: 55,
 62: 56,
 63: 57,
 64: 58,
 65: 59,
 67: 60,
 70: 61,
 72: 62,
 73: 63,
 74: 64,
 75: 65,
 76: 66,
 77: 67,
 78: 68,
 79: 69,
 80: 70,
 81: 71,
 82: 72,
 84: 73,
 85: 74,
 86: 75,
 87: 76,
 88: 77,
 89: 78,
 90: 79}

In [7]:
coco_pairs_path = DATA_PROCESSED / "coco_val_blur_pairs_metadata.csv"
coco_pairs_df = pd.read_csv(coco_pairs_path)

print(len(coco_pairs_df))
coco_pairs_df.head()

5000


,image_id,sharp_path,blur_path,sharp_shape,blur_shape,sharp_lap_var,blur_lap_var,blur_level,blur_ratio
0,139,C:\Users\User\comp6001_assignment1\data\raw\co...,C:\Users\User\comp6001_assignment1\data\raw\co...,"(426, 640)","(426, 640)",854.226167,854.226167,high_blur,1.0
1,285,C:\Users\User\comp6001_assignment1\data\raw\co...,C:\Users\User\comp6001_assignment1\data\raw\co...,"(640, 586)","(640, 586)",4351.070365,4351.070365,low_blur,1.0
2,632,C:\Users\User\comp6001_assignment1\data\raw\co...,C:\Users\User\comp6001_assignment1\data\raw\co...,"(483, 640)","(483, 640)",4177.362686,4177.362686,low_blur,1.0
3,724,C:\Users\User\comp6001_assignment1\data\raw\co...,C:\Users\User\comp6001_assignment1\data\raw\co...,"(500, 375)","(500, 375)",2844.920072,2844.920072,low_blur,1.0
4,776,C:\Users\User\comp6001_assignment1\data\raw\co...,C:\Users\User\comp6001_assignment1\data\raw\co...,"(640, 428)","(640, 428)",4690.731953,4690.731953,low_blur,1.0


In [8]:
sample_n = min(200, len(coco_pairs_df))
sample_df = coco_pairs_df.sample(sample_n, random_state=SEED).reset_index(drop=True)

print(len(sample_df))

200


In [9]:
COCO_RL_DIR = DATA_PROCESSED / "coco_rl_restored"
print(COCO_RL_DIR.exists())

True


In [10]:
def copy_image(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

def convert_bbox_to_yolo(bbox, img_width, img_height):
    x, y, w, h = bbox

    x_center = (x + w / 2) / img_width
    y_center = (y + h / 2) / img_height
    width = w / img_width
    height = h / img_height

    return x_center, y_center, width, height

In [11]:
image_id_to_annotations = {}

for _, ann in annotations_df.iterrows():
    image_id = ann["image_id"]
    if image_id not in image_id_to_annotations:
        image_id_to_annotations[image_id] = []
    image_id_to_annotations[image_id].append(ann)

In [12]:
dataset_rows = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    image_id_str = row["image_id"]
    image_id = int(image_id_str)

    sharp_path = Path(row["sharp_path"])
    blur_path = Path(row["blur_path"])
    rl_path = COCO_RL_DIR / f"{image_id_str}.jpg"

    image_info = images_df[images_df["id"] == image_id]

    if len(image_info) == 0:
        continue

    image_info = image_info.iloc[0]
    img_width = image_info["width"]
    img_height = image_info["height"]
    filename = f"{image_id_str}.jpg"

    anns = image_id_to_annotations.get(image_id, [])

    yolo_lines = []

    for ann in anns:
        if ann.get("iscrowd", 0) == 1:
            continue

        category_id = ann["category_id"]
        bbox = ann["bbox"]

        class_id = category_id_to_index[category_id]
        x_center, y_center, width, height = convert_bbox_to_yolo(
            bbox,
            img_width,
            img_height
        )

        yolo_lines.append(
            f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        )

    if len(yolo_lines) == 0:
        continue

    copy_image(sharp_path, SHARP_IMAGES_DIR / filename)
    copy_image(blur_path, BLUR_IMAGES_DIR / filename)

    if rl_path.exists():
        copy_image(rl_path, RL_IMAGES_DIR / filename)

    with open(SHARP_LABELS_DIR / f"{image_id_str}.txt", "w") as f:
        f.write("\n".join(yolo_lines))

    with open(BLUR_LABELS_DIR / f"{image_id_str}.txt", "w") as f:
        f.write("\n".join(yolo_lines))

    with open(RL_LABELS_DIR / f"{image_id_str}.txt", "w") as f:
        f.write("\n".join(yolo_lines))

    dataset_rows.append({
        "image_id": image_id_str,
        "num_objects": len(yolo_lines),
        "sharp_exists": sharp_path.exists(),
        "blur_exists": blur_path.exists(),
        "rl_exists": rl_path.exists()
    })

  0%|          | 0/200 [00:00<?, ?it/s]

In [13]:
dataset_df = pd.DataFrame(dataset_rows)
dataset_df.head()

,image_id,num_objects,sharp_exists,blur_exists,rl_exists
0,297681,1,True,True,True
1,306733,7,True,True,True
2,125806,2,True,True,True
3,82807,4,True,True,True
4,10977,2,True,True,True


In [14]:
dataset_df.describe()

,image_id,num_objects
count,197.000000,197.000000
mean,279439.045685,6.604061
std,172922.773451,6.416304
min,885.000000,1.000000
25%,127517.000000,2.000000
50%,286708.000000,4.000000
75%,425221.000000,9.000000
max,578093.000000,36.000000


In [15]:
summary = {
    "num_images": int(len(dataset_df)),
    "avg_objects_per_image": float(dataset_df["num_objects"].mean()),
    "sharp_images": int(dataset_df["sharp_exists"].sum()),
    "blur_images": int(dataset_df["blur_exists"].sum()),
    "rl_images": int(dataset_df["rl_exists"].sum())
}

summary

{'num_images': 197,
 'avg_objects_per_image': 6.604060913705584,
 'sharp_images': 197,
 'blur_images': 197,
 'rl_images': 197}

In [16]:
dataset_df.to_csv(OUTPUTS / "tables" / "yolo_dataset_summary.csv", index=False)

In [17]:
with open(OUTPUTS / "logs" / "yolo_dataset_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

In [18]:
sample_image = random.choice(list(SHARP_IMAGES_DIR.glob("*.jpg")))
sample_label = SHARP_LABELS_DIR / f"{sample_image.stem}.txt"

print(sample_image)
print(sample_label)

with open(sample_label, "r") as f:
    print(f.read())

C:\Users\User\comp6001_assignment1\data\processed\yolo_dataset\sharp\images\544444.jpg
C:\Users\User\comp6001_assignment1\data\processed\yolo_dataset\sharp\labels\544444.txt
0 0.572201 0.607367 0.553817 0.402109
30 0.368489 0.821094 0.319087 0.057563


In [19]:
class_names = categories_df.sort_values("id")["name"].tolist()

dataset_yaml = {
    "path": str(YOLO_DATASET_DIR.resolve()),
    "train": "sharp/images",
    "val": "sharp/images",
    "names": {i: name for i, name in enumerate(class_names)}
}

dataset_yaml

{'path': 'C:\\Users\\User\\comp6001_assignment1\\data\\processed\\yolo_dataset',
 'train': 'sharp/images',
 'val': 'sharp/images',
 'names': {0: 'person',
  1: 'bicycle',
  2: 'car',
  3: 'motorcycle',
  4: 'airplane',
  5: 'bus',
  6: 'train',
  7: 'truck',
  8: 'boat',
  9: 'traffic light',
  10: 'fire hydrant',
  11: 'stop sign',
  12: 'parking meter',
  13: 'bench',
  14: 'bird',
  15: 'cat',
  16: 'dog',
  17: 'horse',
  18: 'sheep',
  19: 'cow',
  20: 'elephant',
  21: 'bear',
  22: 'zebra',
  23: 'giraffe',
  24: 'backpack',
  25: 'umbrella',
  26: 'handbag',
  27: 'tie',
  28: 'suitcase',
  29: 'frisbee',
  30: 'skis',
  31: 'snowboard',
  32: 'sports ball',
  33: 'kite',
  34: 'baseball bat',
  35: 'baseball glove',
  36: 'skateboard',
  37: 'surfboard',
  38: 'tennis racket',
  39: 'bottle',
  40: 'wine glass',
  41: 'cup',
  42: 'fork',
  43: 'knife',
  44: 'spoon',
  45: 'bowl',
  46: 'banana',
  47: 'apple',
  48: 'sandwich',
  49: 'orange',
  50: 'broccoli',
  51: 'carrot

In [20]:
import yaml

with open(YOLO_DATASET_DIR / "dataset.yaml", "w") as f:
    yaml.dump(dataset_yaml, f, sort_keys=False)

print(YOLO_DATASET_DIR / "dataset.yaml")

C:\Users\User\comp6001_assignment1\data\processed\yolo_dataset\dataset.yaml
